# SWAPI Incremental Ingestion

Serverless-compatible notebook task. Pulls the top 10 SWAPI people into Bronze and writes modelling/quality governance suggestions.

In [ ]:
from datetime import datetime, timezone
import json
import urllib.request
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

catalog = 'approvalmax_ai_platform'
bronze_schema = 'bronze'
monitoring_schema = 'monitoring'
bronze_table = f'{catalog}.{bronze_schema}.swapi_people_raw'
suggestion_table = f'{catalog}.{monitoring_schema}.swapi_modelling_suggestions'
result_table = f'{catalog}.{monitoring_schema}.great_expectations_results'
run_id = f'swapi_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}'

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {bronze_table} (
  person_url STRING, character_id STRING, name STRING, height STRING, mass STRING,
  hair_color STRING, skin_color STRING, eye_color STRING, birth_year STRING, gender STRING,
  homeworld STRING, films_json STRING, species_json STRING, vehicles_json STRING, starships_json STRING,
  created STRING, edited STRING, source_url STRING, source_system STRING,
  raw_json STRING, ingested_at TIMESTAMP, run_id STRING
) USING DELTA''')

spark.sql(f'''CREATE TABLE IF NOT EXISTS {result_table} (
  validation_run_id STRING, expectation_name STRING, status STRING, severity STRING,
  failed_row_count BIGINT, checked_at TIMESTAMP, details STRING
) USING DELTA''')

def character_id_from_url(url):
    return url.rstrip('/').split('/')[-1]

def fetch_top_10_people():
    request = urllib.request.Request('https://swapi.dev/api/people/?page=1', headers={'User-Agent': 'approvalmax-ai-platform-v2'})
    with urllib.request.urlopen(request, timeout=30) as response:
        payload = json.loads(response.read().decode('utf-8'))
    return payload.get('results', [])[:10]

rows = []
for person in fetch_top_10_people():
    rows.append({
        'person_url': person.get('url'),
        'character_id': character_id_from_url(person.get('url', '')),
        'name': person.get('name'),
        'height': person.get('height'),
        'mass': person.get('mass'),
        'hair_color': person.get('hair_color'),
        'skin_color': person.get('skin_color'),
        'eye_color': person.get('eye_color'),
        'birth_year': person.get('birth_year'),
        'gender': person.get('gender'),
        'homeworld': person.get('homeworld'),
        'films_json': json.dumps(person.get('films') or []),
        'species_json': json.dumps(person.get('species') or []),
        'vehicles_json': json.dumps(person.get('vehicles') or []),
        'starships_json': json.dumps(person.get('starships') or []),
        'created': person.get('created'),
        'edited': person.get('edited'),
        'source_url': 'https://swapi.dev/api/people/?page=1',
        'source_system': 'swapi_dev_people_api',
        'raw_json': json.dumps(person, sort_keys=True),
        'ingested_at': datetime.now(timezone.utc),
        'run_id': run_id,
    })

bronze_schema_def = StructType([
    StructField('person_url', StringType(), True),
    StructField('character_id', StringType(), True),
    StructField('name', StringType(), True),
    StructField('height', StringType(), True),
    StructField('mass', StringType(), True),
    StructField('hair_color', StringType(), True),
    StructField('skin_color', StringType(), True),
    StructField('eye_color', StringType(), True),
    StructField('birth_year', StringType(), True),
    StructField('gender', StringType(), True),
    StructField('homeworld', StringType(), True),
    StructField('films_json', StringType(), True),
    StructField('species_json', StringType(), True),
    StructField('vehicles_json', StringType(), True),
    StructField('starships_json', StringType(), True),
    StructField('created', StringType(), True),
    StructField('edited', StringType(), True),
    StructField('source_url', StringType(), True),
    StructField('source_system', StringType(), True),
    StructField('raw_json', StringType(), True),
    StructField('ingested_at', TimestampType(), True),
    StructField('run_id', StringType(), True),
])

incoming = spark.createDataFrame(rows, schema=bronze_schema_def)
existing_versions = spark.table(bronze_table).select('person_url', 'edited').distinct()
new_rows = incoming.join(existing_versions, ['person_url', 'edited'], 'left_anti')
new_count = new_rows.count()
if new_count:
    new_rows.write.format('delta').mode('append').saveAsTable(bronze_table)

suggestion_rows = [
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'silver', 'object_name': 'swapi_people_current', 'suggestion': 'Current-state Silver table keyed by person_url using latest edited timestamp.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'vault', 'object_name': 'hub_swapi_person', 'suggestion': 'Hub keyed by person_url plus satellite for actor/person attributes.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'gold', 'object_name': 'dim_swapi_actor', 'suggestion': 'Gold dimension for top-10 SWAPI people treated as hypothetical actors.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'gold_enrichment', 'object_name': 'dim_user_actor_enrichment', 'suggestion': 'Optional enrichment of existing user/document marts by reviewed name-matching only; do not redefine financial metrics.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'dbt', 'object_name': 'swapi_people_current_dbt', 'suggestion': 'Generated dbt current-state model after metadata approval.', 'human_review_required': 'true'},
    {'run_id': run_id, 'source_context': 'swapi_people', 'target_layer': 'great_expectations', 'object_name': 'generated_swapi_people_checks', 'suggestion': 'Validate person_url/name/raw payload and primary-key grain after dbt generation.', 'human_review_required': 'true'},
]
suggestion_schema = StructType([
    StructField('run_id', StringType(), False),
    StructField('source_context', StringType(), False),
    StructField('target_layer', StringType(), False),
    StructField('object_name', StringType(), False),
    StructField('suggestion', StringType(), False),
    StructField('human_review_required', StringType(), False),
])
spark.createDataFrame(suggestion_rows, schema=suggestion_schema).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(suggestion_table)

validation_run_id = f'gx_{run_id}'
expectations = {
    'expect_swapi_person_url_not_null': f'SELECT * FROM {bronze_table} WHERE person_url IS NULL',
    'expect_swapi_name_not_null': f'SELECT * FROM {bronze_table} WHERE name IS NULL',
    'expect_swapi_raw_json_not_null': f'SELECT * FROM {bronze_table} WHERE raw_json IS NULL',
    'expect_swapi_source_system_not_null': f'SELECT * FROM {bronze_table} WHERE source_system IS NULL',
    'expect_swapi_person_version_grain': f'SELECT person_url, edited, count(*) AS row_count FROM {bronze_table} GROUP BY person_url, edited HAVING count(*) > 1',
}
result_schema = StructType([
    StructField('validation_run_id', StringType(), False),
    StructField('expectation_name', StringType(), False),
    StructField('status', StringType(), False),
    StructField('severity', StringType(), False),
    StructField('failed_row_count', LongType(), False),
    StructField('checked_at', TimestampType(), False),
    StructField('details', StringType(), True),
])
results = []
critical_failures = []
for name, query in expectations.items():
    failed_count = spark.sql(query).count()
    status = 'passed' if failed_count == 0 else 'failed'
    row = {
        'validation_run_id': validation_run_id,
        'expectation_name': name,
        'status': status,
        'severity': 'critical',
        'failed_row_count': failed_count,
        'checked_at': datetime.now(timezone.utc),
        'details': query,
    }
    results.append(row)
    if failed_count:
        critical_failures.append(row)

spark.createDataFrame(results, schema=result_schema).write.format('delta').mode('append').saveAsTable(result_table)
if critical_failures:
    raise Exception('SWAPI Great Expectations-style validation failed: ' + json.dumps(critical_failures, default=str))

print(f'SWAPI ingestion complete. run_id={run_id}, new_rows={new_count}')
